# Heatmaps: Simulated Condition vs Actual Patient

This notebook computes nearest‑neighbor heatmaps between a simulated condition model
and a real patient model. It renders the heatmap and also saves the points + distances
to disk.


In [ ]:
# Project root resolver
from pathlib import Path

PROJECT_MARKERS = [
    Path('models') / 'condition_effects.json',
    Path('heart_models'),
    Path('data-processing'),
]

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for parent in [cur] + list(cur.parents):
        if all((parent / marker).exists() for marker in PROJECT_MARKERS):
            return parent
    raise FileNotFoundError('Could not locate project root from ' + str(start))

PROJECT_ROOT = find_project_root(Path.cwd())
PROJECT_ROOT


PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026')

In [ ]:
# Config
SIM_DIR = PROJECT_ROOT / 'heart_models' / 'simulated_conditions'
PATIENT_DIR = PROJECT_ROOT / 'heart_models' / 'patient_models'
OUTPUT_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient'
OUTPUT_OBJ_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient_objs'
OUTPUT_CSV_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient_csv'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_OBJ_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient_csv'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_OBJ_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient_objs'
OUTPUT_CSV_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient_csv'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_OBJ_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_DIR = PROJECT_ROOT / 'heart_models' / 'heatmaps_sim_vs_patient_csv'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

PAT_ID = 1
CONDITIONS = ['ASD']  # set to list; use sorted(sim_meshes.keys()) for all

SIM_DIR, PATIENT_DIR, OUTPUT_DIR


(PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/patient_models'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/heatmaps_sim_vs_patient'))

In [ ]:
# Imports
import numpy as np
import re
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go


In [ ]:
# Build condition -> sim path mapping
pattern = re.compile(r'^simulated_(.+)\.obj$', re.IGNORECASE)
sim_meshes = {}
for path in SIM_DIR.glob('*.obj'):
    m = pattern.match(path.name)
    if not m:
        continue
    sim_meshes[m.group(1)] = path

print('Sim conditions:', len(sim_meshes))


Sim conditions: 33


In [ ]:
# Load OBJ vertices
def load_obj_vertices(path: Path):
    verts = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line.startswith('v '):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
    if not verts:
        return np.zeros((0, 3), dtype=np.float32)
    return np.array(verts, dtype=np.float32)


In [ ]:
# ICP (rigid, Kabsch)
def rigid_transform_kabsch(A: np.ndarray, B: np.ndarray):
    assert A.shape == B.shape and A.shape[1] == 3
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = AA.T @ BB
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T
    t = centroid_B - R @ centroid_A
    return R, t

def icp_align(src, tgt, iters=20):
    src = src.copy()
    nbrs = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(tgt)
    for _ in range(iters):
        dists, idx = nbrs.kneighbors(src)
        matched = tgt[idx[:, 0]]
        R, t = rigid_transform_kabsch(src, matched)
        src = (R @ src.T).T + t
    return src


In [ ]:
# Helpers
def center_and_scale(xyz: np.ndarray, method: str = 'rms_radius'):
    xyz = xyz.astype(np.float32)
    c = xyz.mean(axis=0)
    centered = xyz - c
    if method == 'rms_radius':
        r = np.sqrt((centered ** 2).sum(axis=1)).mean()
        s = 1.0 / (r + 1e-8)
    else:
        lo = centered.min(axis=0)
        hi = centered.max(axis=0)
        diag = np.linalg.norm(hi - lo)
        s = 1.0 / (diag + 1e-8)
    return centered * s

def nn_distances(src: np.ndarray, tgt: np.ndarray):
    if src.shape[0] == 0 or tgt.shape[0] == 0:
        return np.array([])
    nbrs = NearestNeighbors(n_neighbors=1, algorithm='auto').fit(tgt)
    dists, idx = nbrs.kneighbors(src)
    return dists[:, 0]

def subsample(pts, n, rng):
    if pts.shape[0] <= n:
        return pts
    idx = rng.choice(pts.shape[0], size=n, replace=False)
    return pts[idx]


In [ ]:
def _colorscale_value(t, colorscale):
    # t in [0,1]
    if t <= colorscale[0][0]:
        return colorscale[0][1]
    if t >= colorscale[-1][0]:
        return colorscale[-1][1]
    for i in range(1, len(colorscale)):
        t0, c0 = colorscale[i-1]
        t1, c1 = colorscale[i]
        if t <= t1:
            u = (t - t0) / (t1 - t0 + 1e-12)
            r0,g0,b0 = c0
            r1,g1,b1 = c1
            return (r0 + (r1 - r0) * u, g0 + (g1 - g0) * u, b0 + (b1 - b0) * u)
    return colorscale[-1][1]

def _to_rgb_tuple(color_str):
    # expects 'rgb(r,g,b)'
    s = color_str.strip().lower()
    s = s.replace('rgb(', '').replace(')', '')
    parts = [float(x) for x in s.split(',')]
    return tuple(parts)

def _build_colorscale():
    # convert HEATMAP_COLORSCALE to numeric tuples
    out = []
    for t, c in HEATMAP_COLORSCALE:
        out.append((t, _to_rgb_tuple(c)))
    return out

def save_pointcloud_obj(path: Path, points: np.ndarray, colors: np.ndarray):
    # OBJ with vertex colors (v x y z r g b)
    lines = []
    for (x, y, z), (r, g, b) in zip(points, colors):
        lines.append(f'v {x:.6f} {y:.6f} {z:.6f} {r:.6f} {g:.6f} {b:.6f}')
    path.write_text('\n'.join(lines))

HEATMAP_COLORSCALE = [
    [0.0, 'rgb(30,136,229)'],
    [0.25, 'rgb(102,187,106)'],
    [0.5, 'rgb(255,193,7)'],
    [0.75, 'rgb(255,152,0)'],
    [1.0, 'rgb(244,67,54)'],
]

def _auto_cmin_cmax(dists):
    if dists.size == 0:
        return 0.0, 1.0
    return float(dists.min()), float(dists.max())

# Render + save heatmap

def render_and_save_heatmap(cond: str, pat_id: int, max_points=50000, point_size=2.0):
    sim_path = sim_meshes.get(cond)
    pat_path = PATIENT_DIR / f'heart_model_pat{pat_id}.obj'
    if sim_path is None or not sim_path.exists():
        print(f'Missing simulated file for {cond}')
        return
    if not pat_path.exists():
        print(f'Missing patient file: {pat_path}')
        return

    sim = load_obj_vertices(sim_path)
    pat = load_obj_vertices(pat_path)
    if sim.shape[0] == 0 or pat.shape[0] == 0:
        print(f'Empty data for {cond} / pat{pat_id}')
        return

    sim = center_and_scale(sim)
    pat = center_and_scale(pat)
    sim_aligned = icp_align(sim, pat)

    rng = np.random.default_rng(0)
    sim_r = subsample(sim_aligned, max_points, rng)
    pat_r = subsample(pat, max_points, rng)
    dists = nn_distances(sim_r, pat_r)
    cmin, cmax = _auto_cmin_cmax(dists)
    cmax = 0.8 * cmax
    cs = _build_colorscale()
    norm = (dists - cmin) / (cmax - cmin + 1e-12)
    colors = np.array([_colorscale_value(t, cs) for t in norm], dtype=np.float32) / 255.0
    cmin, cmax = _auto_cmin_cmax(dists)
    cmax = 0.8 * cmax

    sim_r = sim_r - sim_r.mean(axis=0)
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=sim_r[:,0], y=sim_r[:,1], z=sim_r[:,2], mode='markers',
        name=f'{cond} vs pat{pat_id}',
        marker=dict(size=point_size, color=dists, cmin=cmin, cmax=cmax, colorscale=HEATMAP_COLORSCALE, opacity=0.9, colorbar=dict(title='NN dist'))
    ))
    fig.update_layout(
        title=f'Heatmap: simulated {cond} vs patient {pat_id}',
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=60), height=760, paper_bgcolor='white'
    )
    fig.show()

    # Save heatmap data
    out_obj = OUTPUT_OBJ_DIR / f'simulation_{cond}_patient{pat_id}_heatmap.obj'
    save_pointcloud_obj(out_obj, sim_r, colors)
    print('Saved OBJ:', out_obj)
    out_csv = OUTPUT_CSV_DIR / f'simulation_{cond}_patient{pat_id}_heatmap.csv'
    np.savetxt(str(out_csv), np.column_stack([sim_r, dists]), delimiter=',', header='x,y,z,dist', comments='')
    print('Saved CSV:', out_csv)

def render_overlay_sim_vs_patient(cond: str, pat_id: int, point_size=2.0, max_points=50000):
    sim_path = sim_meshes.get(cond)
    pat_path = PATIENT_DIR / f'heart_model_pat{pat_id}.obj'
    if sim_path is None or not sim_path.exists():
        print(f'Missing simulated file for {cond}')
        return
    if not pat_path.exists():
        print(f'Missing patient file: {pat_path}')
        return

    sim = load_obj_vertices(sim_path)
    pat = load_obj_vertices(pat_path)
    if sim.shape[0] == 0 or pat.shape[0] == 0:
        print(f'Empty data for {cond} / pat{pat_id}')
        return

    sim = center_and_scale(sim)
    pat = center_and_scale(pat)
    sim_aligned = icp_align(sim, pat)

    rng = np.random.default_rng(0)
    sim_r = subsample(sim_aligned, max_points, rng)
    pat_r = subsample(pat, max_points, rng)

    sim_r = sim_r - sim_r.mean(axis=0)
    pat_r = pat_r - pat_r.mean(axis=0)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=pat_r[:,0], y=pat_r[:,1], z=pat_r[:,2], mode='markers',
        name=f'Patient {pat_id}',
        marker=dict(size=point_size, color='rgb(170,170,170)', opacity=0.6)
    ))
    fig.add_trace(go.Scatter3d(
        x=sim_r[:,0], y=sim_r[:,1], z=sim_r[:,2], mode='markers',
        name=f'Simulated {cond}',
        marker=dict(size=point_size, color='rgb(220,40,40)', opacity=0.85)
    ))
    fig.update_layout(
        title=f'Overlay: simulated {cond} vs patient {pat_id}',
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=60), height=760, paper_bgcolor='white'
    )
    fig.show()


In [ ]:
# Run
for cond in CONDITIONS:
    render_overlay_sim_vs_patient(cond, PAT_ID)
    render_and_save_heatmap(cond, PAT_ID)


Saved OBJ: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/heatmaps_sim_vs_patient_objs/simulation_ASD_patient1_heatmap.obj
Saved CSV: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/heatmaps_sim_vs_patient_csv/simulation_ASD_patient1_heatmap.csv
